In [0]:
%restart_python
%pip install boto3
import boto3
import os
from botocore.exceptions import NoCredentialsError
import datetime
import sys
sys.path.insert(0, '/Workspace/Shared')
import etl_helpers 
from pyspark.sql.functions import collect_list, concat_ws, udf, lit
from pyspark.sql.types import IntegerType


runcycleid = etl_helpers.start_run_cycle("factRequestExtensions")

try:
    # Get last successful run time
    df_lastrun = spark.sql(f"""
        SELECT runcyclestartat as createddate 
        FROM dimruncycle 
        WHERE packagename = 'factRequestExtensions' AND success = 't' 
        ORDER BY runcycleid DESC LIMIT 1
    """)
    
    if df_lastrun.count() > 0:
        maxcreatedate_str = df_lastrun.first().createddate.strftime("%Y-%m-%d %H:%M:%S")
    else:
        maxcreatedate_str = "2025-07-11 00:00:00"

    df_existing = spark.sql(f"""
        SELECT DISTINCT foiministryrequestid, foirequest_id
        FROM foi_mod.foiministryrequests
        WHERE created_at > '{maxcreatedate_str}' OR try_cast(updated_at AS DATE) > '{maxcreatedate_str}'
    """)
    
    df_existing = df_existing.localCheckpoint()
    df_existing.createOrReplaceTempView("temp_requests")
    
    change_count = df_existing.count()
    print(f"Records to process: {change_count}")

    if change_count == 0:
        raise Exception("no changes for today")

    spark.sql(f"""
        MERGE INTO default.factRequestExtensions dd
        USING (SELECT foiministryrequestid FROM temp_requests) AS temp
        ON temp.foiministryrequestid = dd.foiministryrequestid
        WHEN MATCHED AND dd.sourceoftruth = 'FOIMOD' THEN
            UPDATE SET dd.activeflag = 'N'
    """)

    query = f"""
        select * from 
        (
            SELECT *
            FROM (
                SELECT 
                    closedate,
                    case
                        when closedate is not null then createdby
                        else null
                    end as completedby,
                    duedate + interval 1 day as extensionactiondate,
                    duedate,
                    recordspagecount,
                    fmr.foirequest_id,
                    fmr.foiministryrequestid,
                    version,
                    ROW_NUMBER() OVER (
                        PARTITION BY fmr.foiministryrequestid 
                        ORDER BY created_at DESC
                    ) AS rn
                FROM foi_mod.foiministryrequests fmr
                INNER JOIN temp_requests tr ON fmr.foiministryrequestid = tr.foiministryrequestid
            ) sub
            WHERE rn = 1
        ) sq

        join 

        (
            select *
            from (
                select
                    foirequestextensionid,
                    extendedduedays,
                    extendedduedate,
                    e.createdby,
                    created_at,
                    extendedduedays,
                    TRY_CAST(approvednoofdays AS INT) as approvednoofdays,
                    updatedby,
                    TRY_CAST(updated_at AS DATE) as updated_at,
                    version,
                    foiministryrequest_id,
                    e.extensionreasonid as extensiontypeid,
                    e.isactive,
                    es.name as extensionstatus,
                    ROW_NUMBER() OVER (
                        PARTITION BY foirequestextensionid 
                        ORDER BY version DESC
                    ) AS rn
                from foi_mod.foirequestextensions e
                join foi_mod.extensionstatuses es on es.extensionstatusid = e.extensionstatusid
            ) sub where rn = 1
        ) sq2 on sq2.foiministryrequest_id = sq.foiministryrequestid

        join 

        (
            select *
            from (
                select
                    TRY_CAST(updated_at AS DATE) as approveddate,
                    foiministryrequest_id,
                    ROW_NUMBER() OVER (
                        PARTITION BY foirequestextensionid 
                        ORDER BY version asc
                    ) AS rn
                from foi_mod.foirequestextensions
            ) sub where rn = 1
        ) sq3 on sq3.foiministryrequest_id = sq.foiministryrequestid
        """

    print(query)

    df = spark.sql(query)
    print(df.count())
    df.show()

    df_mapped = df.selectExpr( 
        "foirequest_id as foirequestid",
        f"{runcycleid} as runcycleid",
        "createdby AS createdby",
        "created_at as createddate",
        "updatedby as modifiedby",
        "updated_at as modifieddate",
        "'' as cstatus",
        "extensiontypeid as extensiontypeid",
        "'' as approvedby",
        "approveddate as approveddate",
        "extendedduedays as extensiondays",
        "extendedduedate as extendeddate",
        "'' AS comments",
        "'' AS approvedcomments",
        "approvednoofdays AS requesteddays",
        "'' AS type",
        "'' AS cnoticetooic",
        "closedate AS completeddate",
        "completedby AS completedby",
        "'' AS completedcomments",
        "'' as ticategory",
        "extensionactiondate AS extensionactiondate",
        "extensionstatus AS approvedstatus",
        "duedate AS oldtargetdate",
        "'' as oldestimateddeliverydate",
        "'' AS consultationtype",
        "recordspagecount as noofpagesdisclosed",
        "recordspagecount as noofpagessent",
        "foirequestextensionid as requestextid",
        "isactive AS activeflag",
        "'FOIMOD' AS sourceoftruth",
        "foiministryrequestid AS foiministryrequestid"
    )
    df_mapped.show()
    df_mapped.write.format("delta").mode("append").option("mergeSchema", "false").insertInto("factrequestextensions")  
    etl_helpers.end_run_cycle(runcycleid, 't', "factRequestExtensions")
except NoCredentialsError:
    print("Credentials not available")
    etl_helpers.end_run_cycle(runcycleid, 'f', "factRequestExtensions", "Credentials not available")
    raise Exception("notebook failed") from e
except Exception as e:    
    if (str(e) == "no changes for today"):
        # print("here")
        etl_helpers.end_run_cycle(runcycleid, 't', "factRequestExtensions")
    else:
        print(f"An error occurred: {e}")    
        etl_helpers.end_run_cycle(runcycleid, 'f', "factRequestExtensions", f"An error occurred: {e}")
        raise Exception("notebook failed") from e